In [12]:
"""
Music genre classification, listener profiling and content-based recommendation.

Pipeline
  1. Load GTZAN feature CSV, engineer extra features, EDA
  2. Genre classification: grouped cross-validation, model comparison, soft-voting ensemble
  3. Valence/arousal proxy, supervised taste embedding, offline retrieval evaluation
  4. Listener profile for a playlist
  5. Recommenders: catalog similarity, iso-principle trajectory, live previews (iTunes Search API)

Usage
  python music_pipeline.py                                   # 30-sec CSV, catalog recommenders
  python music_pipeline.py --csv features_3_sec.csv --segment 3
  python music_pipeline.py --playlist-dir my_playlist --live
"""
import argparse
import os
import re
import shutil
import subprocess
import tempfile
import time
import json
import random
import urllib.parse
import urllib.request
import warnings
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
import numpy as np
import pandas as pd
import seaborn as sns
from joblib import Parallel, delayed
from scipy.stats import entropy, f_oneway, kurtosis, skew, ttest_ind, binomtest
from sklearn.base import clone
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import (ExtraTreesClassifier, HistGradientBoostingClassifier,
                              RandomForestClassifier)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, cohen_kappa_score, confusion_matrix,
                             f1_score, matthews_corrcoef)
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PowerTransformer, RobustScaler, StandardScaler
from sklearn.svm import SVC

warnings.filterwarnings("ignore")

SEED = 42
SR = 22050
OUT_DIR = "outputs"
GENRES = ["blues", "classical", "country", "disco", "hiphop", "jazz", "metal", "pop", "reggae", "rock"]
SCALAR_STATS = ["chroma_stft", "rms", "spectral_centroid", "spectral_bandwidth", "rolloff",
                "zero_crossing_rate", "harmony", "perceptr"]
BASE_COLS = ([f"{s}_{m}" for s in SCALAR_STATS for m in ("mean", "var")] + ["tempo"]
             + [f"mfcc{i}_{m}" for i in range(1, 21) for m in ("mean", "var")])
META_COLS = ["filename", "label", "title", "artist", "language", "preview_url"]
EPS = 1e-8


def savefig(name):
    os.makedirs(OUT_DIR, exist_ok=True)
    path = os.path.join(OUT_DIR, name)
    plt.tight_layout()
    plt.savefig(path, dpi=200)
    plt.close()
    print(f"[saved] {path}")



# 1. FEATURES AND DATA


def find_ffmpeg():
    path = shutil.which("ffmpeg")
    if path:
        return path
    try:
        import imageio_ffmpeg
        return imageio_ffmpeg.get_ffmpeg_exe()
    except Exception:
        return None


def _chunk_features(y, sr):
    """Same feature definitions as the GTZAN CSV, for one audio chunk."""
    import librosa
    harm, perc = librosa.effects.hpss(y)
    tempo, _ = librosa.beat.beat_track(y=y, sr=sr)
    series = {
        "chroma_stft": librosa.feature.chroma_stft(y=y, sr=sr),
        "rms": librosa.feature.rms(y=y),
        "spectral_centroid": librosa.feature.spectral_centroid(y=y, sr=sr),
        "spectral_bandwidth": librosa.feature.spectral_bandwidth(y=y, sr=sr),
        "rolloff": librosa.feature.spectral_rolloff(y=y, sr=sr),
        "zero_crossing_rate": librosa.feature.zero_crossing_rate(y=y),
        "harmony": harm,
        "perceptr": perc,
    }
    feats = {f"{k}_{m}": float(f(v)) for k, v in series.items() for m, f in (("mean", np.mean), ("var", np.var))}
    feats["tempo"] = float(np.atleast_1d(tempo)[0])
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20)
    for i in range(20):
        feats[f"mfcc{i + 1}_mean"] = float(mfcc[i].mean())
        feats[f"mfcc{i + 1}_var"] = float(mfcc[i].var())
    return feats


def extract_audio_features(path, duration=30.0, segment_sec=None):
    """Return a DataFrame with one row per segment (or one row for the whole clip)."""
    import librosa
    tmp = None
    try:
        if path.lower().endswith((".m4a", ".mp4", ".aac")):
            ffmpeg = find_ffmpeg()
            if ffmpeg is None:
                raise RuntimeError("ffmpeg is needed to decode .m4a/.aac files (pip install imageio-ffmpeg)")
            tmp = path + ".wav"
            subprocess.run([ffmpeg, "-y", "-i", path, "-ar", str(SR), "-ac", "1", "-t", str(duration), tmp],
                           stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
            path_to_load = tmp
        else:
            path_to_load = path
        y, sr = librosa.load(path_to_load, sr=SR, duration=duration)
    finally:
        if tmp and os.path.exists(tmp):
            os.remove(tmp)

    if segment_sec:
        n = int(segment_sec * sr)
        chunks = [y[i:i + n] for i in range(0, len(y) - n + 1, n)] or [y]
    else:
        chunks = [y]
    rows = [_chunk_features(c, sr) for c in chunks]
    df = pd.DataFrame(rows)
    stem = os.path.splitext(os.path.basename(path))[0]
    df["song_id"] = stem
    df["filename"] = os.path.basename(path)
    return df


def engineer_features(df):
    """Derived descriptors: HPSS balance, variability, spectral ratios, MFCC shape statistics."""
    df = df.drop(columns=[c for c in df.columns if c.startswith("fe_")]).copy()
    c = lambda name: df[name]
    cv = lambda var, mean: np.sqrt(np.maximum(var, 0)) / (np.abs(mean) + EPS)
    f = {}

    hv, pv = c("harmony_var"), c("perceptr_var")
    f["fe_hpr_log"] = np.log10((hv + EPS) / (pv + EPS))
    f["fe_harmonic_fraction"] = hv / (hv + pv + EPS)
    f["fe_hp_energy_log"] = np.log10(hv + pv + EPS)
    f["fe_percussive_transience"] = pv * c("zero_crossing_rate_mean") / (c("rms_mean") + EPS)
    f["fe_harmonic_warmth"] = hv * c("chroma_stft_mean") / (c("spectral_centroid_mean") + EPS)

    for s, short in [("rms", "rms"), ("spectral_centroid", "centroid"), ("spectral_bandwidth", "bandwidth"),
                     ("rolloff", "rolloff"), ("chroma_stft", "chroma"), ("zero_crossing_rate", "zcr")]:
        f[f"fe_{short}_cv"] = cv(c(f"{s}_var"), c(f"{s}_mean"))

    cent, bw, roll = c("spectral_centroid_mean"), c("spectral_bandwidth_mean"), c("rolloff_mean")
    f["fe_rolloff_to_centroid"] = roll / (cent + EPS)
    f["fe_bw_to_centroid"] = bw / (cent + EPS)
    f["fe_rolloff_to_bw"] = roll / (bw + EPS)
    f["fe_centroid_bw_product"] = np.log(cent * bw + EPS)
    f["fe_brightness_to_roughness"] = np.log1p(cent) / (c("zero_crossing_rate_mean") + 0.01)
    f["fe_zcr_rms_ratio"] = c("zero_crossing_rate_mean") / (c("rms_mean") + EPS)
    f["fe_energy_tempo"] = c("rms_mean") * np.log1p(c("tempo"))
    f["fe_tempo_chroma"] = np.log1p(c("tempo")) * c("chroma_stft_mean")

    M = df[[f"mfcc{i}_mean" for i in range(1, 21)]].values
    V = df[[f"mfcc{i}_var" for i in range(1, 21)]].values
    k = np.arange(1, 21) - 10.5
    low, mid, high = M[:, :5].mean(1), M[:, 5:12].mean(1), M[:, 12:].mean(1)
    f["fe_mfcc_low"], f["fe_mfcc_mid"], f["fe_mfcc_high"] = low, mid, high
    f["fe_mfcc_tilt"] = low - high
    f["fe_mfcc_curvature"] = 2 * mid - low - high
    f["fe_mfcc_slope"] = (M - M.mean(1, keepdims=True)) @ k / (k ** 2).sum()
    f["fe_mfcc_even_odd"] = M[:, 1::2].mean(1) - M[:, 2::2].mean(1)
    f["fe_mfcc_std"] = M.std(1)
    f["fe_mfcc_skew"] = skew(M, axis=1)
    f["fe_mfcc_kurt"] = kurtosis(M, axis=1)
    f["fe_mfcc_abs_mean"] = np.abs(M).mean(1)
    f["fe_mfcc_adjacent_std"] = np.diff(M, axis=1).std(1)
    f["fe_mfcc_volatility"] = np.log1p(V.mean(1))
    f["fe_mfcc_var_tilt"] = np.log1p(V[:, :5].mean(1)) - np.log1p(V[:, 12:].mean(1))
    for i in (1, 2, 3):
        f[f"fe_mfcc{i}_snr"] = np.abs(c(f"mfcc{i}_mean")) / (np.sqrt(np.maximum(c(f"mfcc{i}_var"), 0)) + EPS)
    f["fe_mfcc_delta_12"] = c("mfcc2_mean") - c("mfcc1_mean")
    f["fe_mfcc_delta_23"] = c("mfcc3_mean") - c("mfcc2_mean")

    return pd.concat([df, pd.DataFrame(f, index=df.index)], axis=1)


def feature_columns(df):
    return BASE_COLS + [c for c in df.columns if c.startswith("fe_")]


def to_song_level(df, cols):
    """Average segment rows into one row per song, keeping metadata."""
    g = df.groupby("song_id", sort=False)
    out = g[cols].mean()
    for m in META_COLS:
        if m in df.columns:
            out[m] = g[m].first()
    return out.reset_index()


def make_demo_data(n_per_genre=100):
    """Synthetic stand-in used only when the GTZAN CSV is missing."""
    rng = np.random.default_rng(SEED)
    typical = dict(chroma_stft_mean=.38, chroma_stft_var=.08, rms_mean=.13, rms_var=.003,
                   spectral_centroid_mean=2200, spectral_centroid_var=4e5, spectral_bandwidth_mean=2200,
                   spectral_bandwidth_var=2e5, rolloff_mean=4500, rolloff_var=1.5e6,
                   zero_crossing_rate_mean=.1, zero_crossing_rate_var=.002, harmony_var=.01,
                   perceptr_var=.005, tempo=120)
    shifts = rng.normal(size=(len(GENRES), len(typical)))
    rows = []
    for gi, g in enumerate(GENRES):
        for i in range(n_per_genre):
            row = {c: v * np.exp(0.3 * shifts[gi, j] + rng.normal(0, 0.3)) for j, (c, v) in enumerate(typical.items())}
            row["harmony_mean"] = rng.normal(0, 1e-3)
            row["perceptr_mean"] = rng.normal(0, 1e-3)
            off = rng.normal(0, 8, size=20) * 0 + shifts[gi, :1] * 6
            for m in range(1, 21):
                row[f"mfcc{m}_mean"] = float(rng.normal(off[0] * (m <= 3) - 100 * (m == 1) + 60 * (m == 2), 10 / np.sqrt(m)))
                row[f"mfcc{m}_var"] = float(rng.gamma(4.0, 20.0))
            row.update(label=g, filename=f"{g}.{i:05d}.wav", song_id=f"{g}.{i:05d}")
            rows.append(row)
    return pd.DataFrame(rows)


def load_gtzan(csv_path):
    if not os.path.exists(csv_path):
        print(f"[warn] '{csv_path}' not found; using synthetic demo data (results are not meaningful).")
        raw = make_demo_data()
    else:
        raw = pd.read_csv(csv_path).drop(columns=["length"], errors="ignore")
        raw["song_id"] = raw["filename"].str.extract(r"^([a-z]+\.\d+)")[0]
    n_base = len(BASE_COLS)
    df = engineer_features(raw)
    print(f"[data] {len(df)} rows, {df['song_id'].nunique()} songs, {n_base} base + "
          f"{len(feature_columns(df)) - n_base} engineered features")
    return df



# 2. EDA


def run_eda(songs):
    print("\n EDA ")
    print(songs[["filename", "tempo", "rms_mean", "spectral_centroid_mean", "chroma_stft_mean", "label"]].head())
    print("\nClass balance:\n", songs["label"].value_counts().to_string())
    os.makedirs(OUT_DIR, exist_ok=True)
    songs.to_csv(os.path.join(OUT_DIR, "feature_table.csv"), index=False)

    order = sorted(songs["label"].unique())
    fig, ax = plt.subplots(2, 3, figsize=(20, 11))
    kw = dict(data=songs, x="label", order=order, hue="label", hue_order=order, legend=False)
    sns.boxplot(y="tempo", palette="Set2", ax=ax[0, 0], **kw)
    ax[0, 0].set(title="Tempo by genre", ylabel="BPM")
    sns.violinplot(y="spectral_centroid_mean", palette="coolwarm", ax=ax[0, 1], **kw)
    ax[0, 1].set(title="Spectral centroid (brightness)", ylabel="Hz")
    sns.scatterplot(data=songs, x="spectral_centroid_mean", y="rms_mean", hue="label", hue_order=order,
                    palette="tab10", alpha=.7, s=35, ax=ax[0, 2])
    ax[0, 2].set(title="RMS energy vs spectral centroid", xlabel="Spectral centroid (Hz)", ylabel="RMS")
    ax[0, 2].legend(fontsize=7, ncol=2)
    sns.boxplot(y="fe_hpr_log", palette="Spectral", ax=ax[1, 0], **kw)
    ax[1, 0].axhline(0, color="k", ls="--", lw=1)
    ax[1, 0].set(title="log10 harmonic / percussive variance", ylabel="")
    sns.boxplot(y="zero_crossing_rate_mean", palette="magma", ax=ax[1, 1], **kw)
    ax[1, 1].set(title="Zero-crossing rate", ylabel="")
    cols = [f"mfcc{i}_mean" for i in range(1, 11)]
    z = songs.groupby("label")[cols].mean()
    sns.heatmap((z - z.mean()) / z.std(), cmap="vlag", center=0, ax=ax[1, 2], cbar_kws={"label": "column z-score"})
    ax[1, 2].set(title="Mean MFCC 1-10 per genre (column-standardised)", ylabel="")
    for a in ax.flat[[0, 1, 3, 4]]:
        a.tick_params(axis="x", rotation=45)
    fig.suptitle("Genre EDA: tempo, spectral, harmonic and timbral comparisons", fontsize=15, fontweight="bold")
    savefig("1_eda.png")


# 3. GENRE CLASSIFICATION


def preprocessor():
    return make_pipeline(RobustScaler(), PowerTransformer(method="yeo-johnson"), StandardScaler())


def make_models(n_jobs=1):
    return {
        "SVM (RBF)": SVC(C=5, gamma="scale", probability=True, random_state=SEED),
        "Logistic Regression": LogisticRegression(C=0.5, max_iter=3000),
        "KNN (k=7)": KNeighborsClassifier(7, weights="distance"),
        "LDA (shrinkage)": LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto"),
        "Extra Trees": ExtraTreesClassifier(500, max_features="sqrt", n_jobs=n_jobs, random_state=SEED),
        "Random Forest": RandomForestClassifier(500, max_features="sqrt", n_jobs=n_jobs, random_state=SEED),
        "HistGradientBoosting": HistGradientBoostingClassifier(max_iter=250, learning_rate=0.06,
                                                               l2_regularization=1.5, random_state=SEED),
        "MLP (256-128)": MLPClassifier((256, 128), alpha=0.1, max_iter=800, early_stopping=True, random_state=SEED),
    }


class GenreEnsemble:
    """Soft-voting ensemble; each member has its own preprocessing pipeline."""

    def __init__(self, models, classes):
        self.models, self.classes_ = models, list(classes)

    def fit(self, X, y):
        self.pipes = {n: make_pipeline(preprocessor(), clone(m)).fit(X, y) for n, m in self.models.items()}
        return self

    def predict_proba(self, X):
        return np.mean([p.predict_proba(X) for p in self.pipes.values()], axis=0)

    def predict(self, X):
        return self.predict_proba(X).argmax(1)

    def tree_importances(self, feature_cols, top=15):
        for name in ("Random Forest", "Extra Trees"):
            if name in self.pipes:
                imp = pd.Series(self.pipes[name][-1].feature_importances_, index=feature_cols)
                return imp.sort_values(ascending=False).head(top)
        return None


def _fit_fold(name, model, X, y, tr, te):
    pipe = make_pipeline(preprocessor(), clone(model)).fit(X[tr], y[tr])
    return name, te, pipe.predict_proba(X[te])


def song_level(P, y, groups):
    Pg = pd.DataFrame(P).groupby(groups).mean()
    yg = pd.Series(y).groupby(groups).first().loc[Pg.index]
    return Pg.values, yg.values


def scores(P, y, groups):
    Ps, ys = song_level(P, y, groups)
    pred = Ps.argmax(1)
    return dict(acc=accuracy_score(ys, pred), f1=f1_score(ys, pred, average="macro"),
                kappa=cohen_kappa_score(ys, pred), mcc=matthews_corrcoef(ys, pred))


def benchmark_genre_models(df, feature_cols, n_splits=5, top_k=4):
    print("\n GENRE CLASSIFICATION (grouped CV, no song appears in both train and test) ")
    X = df[feature_cols].values
    classes = sorted(df["label"].unique())
    y = pd.Categorical(df["label"], categories=classes).codes
    groups = df["song_id"].values
    folds = list(StratifiedGroupKFold(n_splits, shuffle=True, random_state=SEED).split(X, y, groups))

    models = make_models()
    jobs = [(n, m, tr, te) for n, m in models.items() for tr, te in folds]
    results = Parallel(n_jobs=-1)(delayed(_fit_fold)(n, m, X, y, tr, te) for n, m, tr, te in jobs)
    oof = {n: np.zeros((len(y), len(classes))) for n in models}
    for n, te, P in results:
        oof[n][te] = P

    def row(name, P):
        s = scores(P, y, groups)
        fold_acc = [accuracy_score(y[te], P[te].argmax(1)) for _, te in folds]
        return {"Model": name, "Accuracy": s["acc"], "Macro F1": s["f1"], "Kappa": s["kappa"],
                "MCC": s["mcc"], "Fold std": np.std(fold_acc)}

    table = pd.DataFrame([row(n, P) for n, P in oof.items()]).sort_values("Accuracy", ascending=False)
    members = list(table["Model"].head(top_k))
    oof_ens = np.mean([oof[n] for n in members], axis=0)
    ens_name = f"Soft voting ({', '.join(members)})"
    table = pd.concat([pd.DataFrame([row(ens_name, oof_ens)]), table], ignore_index=True)
    table = table.sort_values("Accuracy", ascending=False).reset_index(drop=True)
    print(table.round(4).to_string(index=False))

    final = GenreEnsemble({n: make_models(-1)[n] for n in members}, classes).fit(X, y)
    final.all_models = GenreEnsemble({n: make_models(-1)[n] for n in ("Random Forest",)}, classes).fit(X, y)

    Ps, ys = song_level(oof_ens, y, groups)
    cm = confusion_matrix(ys, Ps.argmax(1))
    report_confusions(cm, classes, df, groups)
    plot_benchmark(cm, classes, table, ens_name)
    return final, classes, table, (Ps, ys)


def report_confusions(cm, classes, df, groups):
    off = cm.copy()
    np.fill_diagonal(off, 0)
    pairs = [np.unravel_index(i, off.shape) for i in np.argsort(off, axis=None)[::-1][:3]]
    song = df.groupby("song_id").first()
    key = ["tempo", "spectral_centroid_mean", "zero_crossing_rate_mean", "rms_mean", "chroma_stft_mean", "fe_hpr_log"]
    z = (song[key] - song[key].mean()) / song[key].std()
    z["label"] = song["label"]
    means = z.groupby("label").mean()
    print("\nMost confused genre pairs (true -> predicted):")
    for i, j in pairs:
        if off[i, j] == 0:
            continue
        diff = (means.loc[classes[i]] - means.loc[classes[j]]).abs().sort_values()
        print(f"  {classes[i]} -> {classes[j]}: {off[i, j]} songs; closest on "
              f"{diff.index[0]}, {diff.index[1]} (genre-mean z-score gap {diff.iloc[0]:.2f}, {diff.iloc[1]:.2f})")


def plot_benchmark(cm, classes, table, ens_name):
    fig, ax = plt.subplots(1, 2, figsize=(19, 7.5), gridspec_kw={"width_ratios": [1, 1.1]})
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=classes, yticklabels=classes, ax=ax[0])
    best = table.iloc[0]
    ax[0].set(title=f"Out-of-fold confusion matrix (song level)\naccuracy {best['Accuracy']:.1%}, macro F1 {best['Macro F1']:.1%}",
              xlabel="Predicted", ylabel="Actual")
    t = table.copy()
    t["Model"] = t["Model"].str.replace(r"^Soft voting.*", "Soft-voting ensemble", regex=True)
    x = np.arange(len(t))
    ax[1].bar(x - .2, t["Accuracy"] * 100, .4, yerr=t["Fold std"] * 100, capsize=3, label="Accuracy ± fold std")
    ax[1].bar(x + .2, t["Macro F1"] * 100, .4, label="Macro F1")
    ax[1].set_xticks(x)
    ax[1].set_xticklabels(t["Model"], rotation=30, ha="right")
    ax[1].set(ylim=(0, 100), ylabel="%", title="Model comparison (5-fold grouped CV)")
    ax[1].legend(loc="lower left")
    ax[1].grid(axis="y", ls="--", alpha=.4)
    savefig("2_genre_models.png")



# 4. AFFECT PROXY AND TASTE EMBEDDING


class AffectModel:
    """
    Valence/arousal proxy: fixed weighted sums of standardised acoustic features.
    This is a heuristic, not a model trained on listener ratings (e.g. DEAM); treat the
    values as relative positions within a catalog, not absolute emotion labels.
    """
    AROUSAL = {"rms": .40, "centroid": .30, "tempo": .18, "percussive": .12}
    VALENCE = {"mfcc2": .30, "chroma": .25, "harmonic": .25, "centroid": -.20, "zcr": -.25}

    @staticmethod
    def _terms(df):
        return pd.DataFrame({
            "rms": np.log(df["rms_mean"] + EPS), "centroid": np.log(df["spectral_centroid_mean"]),
            "tempo": np.log(df["tempo"]), "percussive": np.log10(df["perceptr_var"] + EPS),
            "mfcc2": df["mfcc2_mean"], "chroma": df["chroma_stft_mean"],
            "harmonic": df["fe_harmonic_fraction"], "zcr": df["zero_crossing_rate_mean"],
        })

    def _raw(self, df):
        z = (self._terms(df) - self.mu) / self.sd
        a = sum(w * z[k] for k, w in self.AROUSAL.items())
        v = sum(w * z[k] for k, w in self.VALENCE.items())
        return v.values, a.values

    def fit(self, df):
        t = self._terms(df)
        self.mu, self.sd = t.mean(), t.std() + EPS
        v, a = self._raw(df)
        self.a_mu = a.mean()
        ac = a - self.a_mu
        self.proj = np.dot(v - v.mean(), ac) / (np.dot(ac, ac) + EPS)   # remove arousal component from valence
        v_o = (v - v.mean()) - self.proj * ac
        self.v_mu, self.v_sd, self.a_sd = v.mean(), v_o.std() + EPS, a.std() + EPS
        return self

    def transform(self, df):
        v, a = self._raw(df)
        ac = a - self.a_mu
        v_o = (v - self.v_mu) - self.proj * ac
        return np.tanh(v_o / self.v_sd * .7), np.tanh(ac / self.a_sd * .7)


def mood_label(v, a):
    if a >= 0:
        return "Energetic / positive" if v >= 0 else "Tense / intense"
    return "Calm / warm" if v >= 0 else "Sad / low-energy"


class TasteEmbedder:
    """Supervised (LDA) timbre space plus valence/arousal; cosine similarity in this space drives recommendations."""

    def __init__(self, feature_cols, affect, va_weight=1.0):
        self.cols, self.affect, self.va_weight = feature_cols, affect, va_weight

    def fit(self, songs):
        self.pre = preprocessor().fit(songs[self.cols].values)
        Z = self.pre.transform(songs[self.cols].values)
        self.lda = LinearDiscriminantAnalysis(solver="eigen", shrinkage="auto").fit(Z, songs["label"].values)
        T = self.lda.transform(Z)
        self.mu, self.sd = T.mean(0), T.std(0) + EPS
        return self

    def transform(self, df):
        T = (self.lda.transform(self.pre.transform(df[self.cols].values)) - self.mu) / self.sd
        v, a = self.affect.transform(df)
        E = np.hstack([T, self.va_weight * np.column_stack([v, a])])
        return E / (np.linalg.norm(E, axis=1, keepdims=True) + EPS)


def evaluate_retrieval(songs, feature_cols, affect, k=5, n_splits=5):
    """Genre precision@k when the query and its neighbours come from held-out songs."""
    print("\n RETRIEVAL EVALUATION (precision@%d on held-out songs; same-genre = relevant) " % k)
    res = {"Raw scaled features, cosine": [], "Taste embedding (LDA + valence/arousal)": []}
    labels = songs["label"].values
    for tr, te in StratifiedKFold(n_splits, shuffle=True, random_state=SEED).split(songs, labels):
        train, test = songs.iloc[tr], songs.iloc[te]
        emb = TasteEmbedder(feature_cols, affect).fit(train)
        Z = emb.pre.transform(test[feature_cols].values)
        Z = Z / np.linalg.norm(Z, axis=1, keepdims=True)
        spaces = {"Raw scaled features, cosine": Z, "Taste embedding (LDA + valence/arousal)": emb.transform(test)}
        y = test["label"].to_numpy()
        for name, E in spaces.items():
            S = E @ E.T
            np.fill_diagonal(S, -np.inf)
            nn = np.argsort(-S, axis=1)[:, :k]
            res[name].append((y[nn] == y[:, None]).mean())
    for name, v in res.items():
        print(f"  {name:<42} P@{k} = {np.mean(v):.3f} ± {np.std(v):.3f}")
    return {n: float(np.mean(v)) for n, v in res.items()}


def statistical_audit(songs, acc_pair):
    print("\n STATISTICAL CHECKS ")
    a = [g["arousal"].values for _, g in songs.groupby("label")]
    F, p = f_oneway(*a)
    print(f"  Arousal differs across genres: ANOVA F={F:.1f}, p={p:.2e}")
    if {"metal", "classical"} <= set(songs["label"]):
        t, p = ttest_ind(songs.loc[songs.label == "metal", "arousal"],
                         songs.loc[songs.label == "classical", "arousal"], equal_var=False)
        print(f"  Metal vs classical arousal: Welch t={t:.1f}, p={p:.2e}")
    r = np.corrcoef(songs["valence"], songs["arousal"])[0, 1]
    print(f"  Valence-arousal correlation after orthogonalisation: r={r:+.3f}")
    Ps, ys = acc_pair
    k, n = int((Ps.argmax(1) == ys).sum()), len(ys)
    print(f"  Classifier vs chance: {k}/{n} correct, exact binomial p={binomtest(k, n, 1 / Ps.shape[1], alternative='greater').pvalue:.2e}")


def plot_validation(songs, ensemble, feature_cols):
    fig, ax = plt.subplots(1, 3, figsize=(20, 6))
    sns.kdeplot(data=songs, x="valence", y="arousal", fill=True, cmap="Blues", thresh=.05, ax=ax[0])
    ax[0].scatter(songs["valence"], songs["arousal"], s=6, alpha=.25, c="navy")
    ax[0].set(title="Valence-arousal density", xlim=(-1.05, 1.05), ylim=(-1.05, 1.05))
    st = songs.groupby("label")[["valence", "arousal"]].agg(["mean", "std"])
    pal = sns.color_palette("tab10", len(st))
    for i, (g, r) in enumerate(st.iterrows()):
        ax[1].errorbar(r[("valence", "mean")], r[("arousal", "mean")], xerr=r[("valence", "std")] / 2,
                       yerr=r[("arousal", "std")] / 2, fmt="o", color=pal[i], capsize=3)
        ax[1].annotate(g, (r[("valence", "mean")], r[("arousal", "mean")]), xytext=(5, 5),
                       textcoords="offset points", fontsize=8.5)
    ax[1].set(title="Genre centroids (± 0.5 sd)", xlim=(-1.05, 1.05), ylim=(-1.05, 1.05), xlabel="Valence", ylabel="Arousal")
    for a in ax[:2]:
        a.axhline(0, color="gray", ls="--", lw=1)
        a.axvline(0, color="gray", ls="--", lw=1)
    imp = ensemble.all_models.tree_importances(feature_cols)
    sns.barplot(x=imp.values, y=imp.index, ax=ax[2], color="#2a7f8f")
    ax[2].set(title="Top 15 features (Random Forest importance)", xlabel="Gini importance")
    savefig("3_validation.png")



# 5. LISTENER PROFILE


@dataclass
class Bundle:
    clf: GenreEnsemble
    affect: AffectModel
    embedder: TasteEmbedder
    feature_cols: list
    catalog: pd.DataFrame
    classes: list

    @property
    def pcols(self):
        return [f"p_{c}" for c in self.classes]

    def enrich(self, raw):
        """Raw feature rows (segments or songs) -> song-level table with genre probabilities and affect."""
        df = engineer_features(raw)
        df[self.pcols] = self.clf.predict_proba(df[self.feature_cols].values)
        songs = to_song_level(df, self.feature_cols + self.pcols)
        songs["valence"], songs["arousal"] = self.affect.transform(songs)
        songs["predicted_genre"] = np.array(self.classes)[songs[self.pcols].values.argmax(1)]
        songs["mood"] = [mood_label(v, a) for v, a in zip(songs["valence"], songs["arousal"])]
        return songs


@dataclass
class Profile:
    name: str
    songs: pd.DataFrame
    genre_probs: np.ndarray
    diversity: float
    gmm: GaussianMixture
    weights: np.ndarray
    emb: np.ndarray


def build_profile(songs, bundle, name):
    p = songs[bundle.pcols].mean().values
    diversity = float(entropy(p, base=2) / np.log2(len(p)))
    va = songs[["valence", "arousal"]].values
    max_k = max(1, min(3, len(songs) // 4))
    fits = [GaussianMixture(k, covariance_type="full", reg_covar=1e-2, n_init=5, random_state=SEED).fit(va)
            for k in range(1, max_k + 1)]
    gmm = min(fits, key=lambda g: g.bic(va))
    songs = songs.copy()
    songs["mood_cluster"] = gmm.predict(va)
    prof = Profile(name, songs, p, diversity, gmm, gmm.predict_proba(va).mean(0), bundle.embedder.transform(songs))
    print_profile(prof, bundle)
    plot_profile(prof, bundle)
    return prof


def print_profile(p, bundle):
    s = p.songs
    top = np.argsort(p.genre_probs)[::-1][:3]
    print(f"\n LISTENING PROFILE: {p.name} ({len(s)} tracks) ")
    print("  Genre mix       : " + ", ".join(f"{bundle.classes[i]} {p.genre_probs[i]:.0%}" for i in top))
    print(f"  Genre diversity : {p.diversity:.2f} (0 = one genre, 1 = evenly spread)")
    print(f"  Tempo           : {s['tempo'].mean():.0f} BPM (sd {s['tempo'].std():.0f})")
    print(f"  Valence / arousal: {s['valence'].mean():+.2f} / {s['arousal'].mean():+.2f} "
          f"(sd {s['valence'].std():.2f} / {s['arousal'].std():.2f})")
    print(f"  Mood clusters   : {p.gmm.n_components}")
    for c in range(p.gmm.n_components):
        v, a = p.gmm.means_[c]
        sub = s[s["mood_cluster"] == c]
        lead = sub["predicted_genre"].mode().iloc[0] if len(sub) else "-"
        print(f"    cluster {c}: weight {p.weights[c]:.0%}, centre ({v:+.2f}, {a:+.2f}), "
              f"{mood_label(v, a)}, mostly {lead}")
    print(f"  Highest energy  : {s.loc[s['arousal'].idxmax(), 'filename']}")
    print(f"  Calmest         : {s.loc[s['arousal'].idxmin(), 'filename']}")


def draw_clusters(ax, gmm, colors, alpha=.15):
    for i in range(gmm.n_components):
        vals, vecs = np.linalg.eigh(gmm.covariances_[i])
        o = vals.argsort()[::-1]
        vals, vecs = vals[o], vecs[:, o]
        ang = np.degrees(np.arctan2(*vecs[:, 0][::-1]))
        w, h = 4 * np.sqrt(np.maximum(vals, 1e-6))
        ax.add_patch(Ellipse(gmm.means_[i], w, h, angle=ang, fc=colors[i], ec=colors[i], alpha=alpha, lw=2))


def plot_profile(p, bundle):
    s, pal = p.songs, sns.color_palette("Set1", 9)
    fig, ax = plt.subplots(2, 2, figsize=(16, 12))
    a = ax[0, 0]
    draw_clusters(a, p.gmm, pal)
    sns.scatterplot(data=s, x="valence", y="arousal", hue="mood_cluster", style="predicted_genre",
                    palette=pal[:p.gmm.n_components], s=110, ax=a)
    a.legend(loc="center left", bbox_to_anchor=(1.01, .5), fontsize=8)
    a.axhline(0, color="gray", ls="--"); a.axvline(0, color="gray", ls="--")
    a.set(xlim=(-1.1, 1.1), ylim=(-1.1, 1.1), title="Valence-arousal map with mood clusters (2 sd)")
    order = np.argsort(p.genre_probs)
    ax[0, 1].barh(np.array(bundle.classes)[order], p.genre_probs[order] * 100, color="#2a7f8f")
    ax[0, 1].set(title=f"Mean genre probability (diversity {p.diversity:.2f})", xlabel="%")
    key = {"tempo": "Tempo", "rms_mean": "Loudness", "spectral_centroid_mean": "Brightness",
           "zero_crossing_rate_mean": "Roughness", "chroma_stft_mean": "Chroma", "fe_harmonic_fraction": "Harmonic share"}
    cat = bundle.catalog[list(key)]
    z = ((s[list(key)].mean() - cat.mean()) / cat.std()).rename(key)
    ax[1, 0].barh(z.index, z.values, color=np.where(z.values >= 0, "#c44e52", "#4c72b0"))
    ax[1, 0].axvline(0, color="k", lw=1)
    ax[1, 0].set(title="Playlist vs catalog average (z-score)", xlabel="standard deviations")
    t = np.arange(1, len(s) + 1)
    ax[1, 1].plot(t, s["arousal"], "o-", color="#d62728", label="Arousal")
    ax[1, 1].plot(t, s["valence"], "s-", color="#1f77b4", label="Valence")
    ax[1, 1].axhline(0, color="gray", ls="--")
    ax[1, 1].set(ylim=(-1.1, 1.1), xlabel="Track order", title="Track-by-track trajectory")
    ax[1, 1].legend()
    fig.suptitle(f"Listening profile: {p.name}", fontsize=15, fontweight="bold")
    savefig("4_profile_dashboard.png")



# 6. RECOMMENDERS

def similar_songs(bundle, seed_idx, k=5):
    """Rubric extension: cosine-similarity song recommender on the taste embedding."""
    cat = bundle.catalog
    E = bundle.embedder.transform(cat)
    sim = E @ E[seed_idx]
    sim[seed_idx] = -np.inf
    idx = np.argsort(-sim)[:k]
    out = cat.iloc[idx][["filename", "label", "valence", "arousal"]].copy()
    out["cosine_similarity"] = sim[idx].round(3)
    return out


def mood_trajectory(bundle, seed_idx, target=(0.65, -0.25), steps=5):
    """Iso-principle path: start near the seed's mood, then step toward a target mood while keeping timbre close."""
    cat = bundle.catalog.reset_index(drop=True)
    E = bundle.embedder.transform(cat)
    sim = E @ E[seed_idx]
    v0, a0 = cat.loc[seed_idx, ["valence", "arousal"]]
    wv, wa = np.linspace(v0, target[0], steps), np.linspace(v0 * 0 + a0, target[1], steps)
    used, path = {seed_idx}, [seed_idx]
    for i in range(1, steps):
        d = np.hypot(cat["valence"].values - wv[i], cat["arousal"].values - wa[i])
        score = .4 * sim - .6 * d
        score[list(used)] = -np.inf
        j = int(np.argmax(score))
        used.add(j); path.append(j)
    out = cat.iloc[path][["filename", "label", "tempo", "valence", "arousal"]].copy()
    out.insert(0, "step", range(1, steps + 1))
    out["timbre_sim"] = sim[path].round(3)
    plot_trajectory(cat, out)
    return out.round(3)


def plot_trajectory(cat, tr):
    fig, ax = plt.subplots(1, 2, figsize=(16, 6.5))
    ax[0].scatter(cat["valence"], cat["arousal"], c="lightgray", s=18, alpha=.4)
    ax[0].plot(tr["valence"], tr["arousal"], "o-", color="crimson", lw=2.5, ms=9)
    for _, r in tr.iterrows():
        ax[0].annotate(f"{r['step']}", (r["valence"], r["arousal"]), xytext=(7, 7), textcoords="offset points",
                       fontweight="bold")
    ax[0].axhline(0, color="gray", ls="--"); ax[0].axvline(0, color="gray", ls="--")
    ax[0].set(xlim=(-1.1, 1.1), ylim=(-1.1, 1.1), xlabel="Valence", ylabel="Arousal",
              title="Mood path through the catalog (numbers = step)")
    ax[1].plot(tr["step"], tr["valence"], "s-", color="seagreen", label="Valence")
    ax[1].plot(tr["step"], tr["arousal"], "o-", color="crimson", label="Arousal")
    ax[1].plot(tr["step"], tr["timbre_sim"], "^--", color="royalblue", label="Timbre similarity to step 1")
    ax[1].axhline(0, color="gray", ls=":")
    ax[1].set(xticks=tr["step"], xlabel="Step", title="Mood shift vs timbre continuity")
    for _, r in tr.iterrows():
        ax[1].annotate(r["filename"].replace(".wav", ""), (r["step"], r["valence"]), xytext=(0, -14),
                       textcoords="offset points", fontsize=7, ha="center")
    ax[1].legend()
    savefig("6_mood_trajectory.png")


def rank01(x):
    return pd.Series(x).rank(pct=True).values


def rank_candidates(cands, profile, bundle, n_recs=10, lam=.7, max_per_artist=2, dup_threshold=.985):
    """Relevance (taste + mood + genre) re-ranked with MMR diversity and cluster-calibrated mood coverage."""
    E = bundle.embedder.transform(cands)
    S = E @ profile.emb.T
    taste = np.sort(S, axis=1)[:, -min(3, S.shape[1]):].mean(1)
    va = cands[["valence", "arousal"]].values
    aff = np.zeros(len(cands))
    for w, mu, cov in zip(profile.weights, profile.gmm.means_, profile.gmm.covariances_):
        d = va - mu
        m = np.einsum("ij,jk,ik->i", d, np.linalg.inv(cov + .03 * np.eye(2)), d)
        aff += w * np.exp(-.5 * m)
    gp, ug = cands[bundle.pcols].values, profile.genre_probs
    genre = gp @ ug / (np.linalg.norm(gp, axis=1) * np.linalg.norm(ug) + EPS)
    rel = .5 * rank01(taste) + .3 * rank01(aff) + .2 * rank01(genre)
    rel[S.max(1) > dup_threshold] = -np.inf            # near-copies of tracks the user already has
    resp = profile.gmm.predict_proba(va)
    CE = E @ E.T

    chosen, artist_n, soft = [], {}, np.zeros(profile.gmm.n_components)
    for _ in range(min(n_recs, len(cands))):
        deficit = profile.weights - (soft / max(len(chosen), 1) if chosen else 0)
        red = CE[:, chosen].max(1) if chosen else np.zeros(len(cands))
        score = lam * rel - (1 - lam) * red + .15 * resp @ deficit
        for i in chosen:
            score[i] = -np.inf
        for i, a in enumerate(cands["artist"].values):
            if artist_n.get(a, 0) >= max_per_artist:
                score[i] = -np.inf
        j = int(np.argmax(score))
        if not np.isfinite(score[j]):
            break
        chosen.append(j)
        soft += resp[j]
        artist_n[cands["artist"].iloc[j]] = artist_n.get(cands["artist"].iloc[j], 0) + 1
    out = cands.iloc[chosen].copy()
    out["match"] = (rel[chosen] * 100).round(1)
    out["nearest_user_track"] = profile.songs["filename"].values[S[chosen].argmax(1)]
    return out.reset_index(drop=True)


class LiveRecommender:
    """Searches the iTunes Search API, analyses real 30-sec previews, and ranks them against a profile."""

    INDIAN_LANGUAGES = ["Hindi", "Punjabi", "Tamil", "Telugu", "Malayalam", "Kannada", "Bengali", "Marathi"]
    GENRE_LANG = {"bollywood": "Hindi", "hindi": "Hindi", "punjabi": "Punjabi", "tamil": "Tamil", "telugu": "Telugu",
                  "malayalam": "Malayalam", "kannada": "Kannada", "bengali": "Bengali", "marathi": "Marathi",
                  "gujarati": "Gujarati", "indian": None, "desi": None, "sufi": None, "ghazal": None,
                  "devotional": None, "carnatic": None, "hindustani": None}
    MOOD_TERMS = {"Energetic / positive": ["upbeat dance", "party", "feel good", "dance hits"],
                  "Tense / intense": ["high energy", "rap", "workout", "rock"],
                  "Sad / low-energy": ["sad songs", "heartbreak", "unplugged", "ghazal"],
                  "Calm / warm": ["chill", "acoustic", "soothing", "lofi", "instrumental"]}
    GENRE_TERMS = {"classical": "classical", "rock": "rock", "pop": "pop", "hiphop": "hip hop", "metal": "rock",
                   "jazz": "fusion", "blues": "soulful", "country": "folk", "disco": "dance", "reggae": "indie"}

    def __init__(self, bundle, languages=None, indian_only=True, segment_sec=None, verify_ssl=True):
        if find_ffmpeg() is None:
            raise RuntimeError("ffmpeg not found. Install it, or run: pip install imageio-ffmpeg")
        self.bundle, self.languages = bundle, languages or self.INDIAN_LANGUAGES
        self.indian_only, self.segment_sec = indian_only, segment_sec
        import ssl
        self.ctx = ssl.create_default_context() if verify_ssl else ssl._create_unverified_context()

    def build_queries(self, profile, n=16, seed=None):
        rng = random.Random(seed if seed is not None else int(time.time()))
        s = profile.songs
        queries = []
        for i in range(n):
            c = int(rng.choices(range(profile.gmm.n_components), weights=profile.weights)[0])
            mood = mood_label(*profile.gmm.means_[c])
            sub = s[s["mood_cluster"] == c]
            genre = sub["predicted_genre"].mode().iloc[0] if len(sub) else "pop"
            lang = self.languages[i % len(self.languages)]
            term = rng.choice(self.MOOD_TERMS[mood]) if i % 2 == 0 else self.GENRE_TERMS.get(genre, "melody")
            queries.append((f"{lang} {term} songs", lang))
        return queries

    def search(self, query, lang, limit=25):
        url = ("https://itunes.apple.com/search?term=" + urllib.parse.quote(query) +
               f"&country=IN&media=music&entity=song&limit={limit}")
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=10, context=self.ctx) as r:
                items = json.loads(r.read().decode("utf-8", "ignore")).get("results", [])
        except Exception:
            return []
        out = []
        for it in items:
            if not it.get("previewUrl"):
                continue
            genre = str(it.get("primaryGenreName", "")).lower()
            detected = next((v or lang for k, v in self.GENRE_LANG.items() if k in genre), None)
            if self.indian_only and detected is None:
                continue
            out.append(dict(title=it.get("trackName", ""), artist=it.get("artistName", ""),
                            language=detected or lang, preview_url=it["previewUrl"], itunes_genre=it.get("primaryGenreName")))
        return out

    def analyse(self, cand):
        tmp = None
        try:
            req = urllib.request.Request(cand["preview_url"], headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=15, context=self.ctx) as r:
                data = r.read()
            ext = ".m4a" if ".m4a" in cand["preview_url"] else ".mp3"
            with tempfile.NamedTemporaryFile(suffix=ext, delete=False) as f:
                f.write(data)
                tmp = f.name
            df = extract_audio_features(tmp, duration=30, segment_sec=self.segment_sec)
            sid = f"{cand['title']} - {cand['artist']}"
            df["song_id"], df["filename"] = sid, sid
            for k in ("title", "artist", "language", "preview_url"):
                df[k] = cand[k]
            return df
        except Exception:
            return None            # undecodable previews are skipped, never imputed
        finally:
            if tmp and os.path.exists(tmp):
                os.remove(tmp)

    def recommend(self, profile, n_recs=10, max_candidates=60, seed=None):
        print(f"\n LIVE RECOMMENDATIONS for {profile.name} ")
        queries = self.build_queries(profile, seed=seed)
        for q, _ in queries[:5]:
            print(f"  query: {q}")
        with ThreadPoolExecutor(8) as ex:
            found = [c for res in ex.map(lambda q: self.search(*q), queries) for c in res]
        seen, cands = set(), []
        have = " ".join(profile.songs["filename"].astype(str)).lower()
        for c in found:
            key = (re.sub(r"\W+", "", c["title"].lower()), re.sub(r"\W+", "", c["artist"].lower()))
            if key in seen or not key[0] or key[0] in have:
                continue
            seen.add(key)
            cands.append(c)
        random.Random(seed).shuffle(cands)
        cands = cands[:max_candidates]
        print(f"  {len(found)} search hits -> {len(cands)} unique candidates; analysing previews...")
        with ThreadPoolExecutor(6) as ex:
            frames = [f for f in ex.map(self.analyse, cands) if f is not None]
        if not frames:
            raise RuntimeError("No previews could be fetched/decoded. Check internet access and ffmpeg.")
        print(f"  decoded {len(frames)} previews")
        pool = self.bundle.enrich(pd.concat(frames, ignore_index=True))
        recs = rank_candidates(pool, profile, self.bundle, n_recs=n_recs)
        recs.insert(0, "rank", np.arange(1, len(recs) + 1))
        show = recs[["rank", "title", "artist", "language", "predicted_genre", "valence", "arousal", "tempo", "match"]]
        print(show.round(2).to_string(index=False))
        print("  (match = relevance percentile within this candidate pool, not an absolute probability)")
        self.plot(profile, recs)
        return recs

    @staticmethod
    def plot(profile, recs):
        fig, ax = plt.subplots(1, 2, figsize=(18, 7.5), gridspec_kw={"width_ratios": [1, 1.1]})
        a = ax[0]
        draw_clusters(a, profile.gmm, sns.color_palette("Set1", 9))
        a.scatter(profile.songs["valence"], profile.songs["arousal"], c="lightgray", edgecolors="dimgray", s=70, label="Your tracks")
        sns.scatterplot(data=recs, x="valence", y="arousal", hue="language", s=150, palette="tab10", ax=a)
        for _, r in recs.iterrows():
            a.annotate(str(r["rank"]), (r["valence"], r["arousal"]), xytext=(6, 5), textcoords="offset points", fontweight="bold")
        a.axhline(0, color="gray", ls="--"); a.axvline(0, color="gray", ls="--")
        a.set(xlim=(-1.1, 1.1), ylim=(-1.1, 1.1), xlabel="Valence", ylabel="Arousal", title="Recommendations on your mood map")
        a.legend(fontsize=8, loc="lower left")
        labels = [f"#{r['rank']} {r['title'][:28]} - {r['artist'][:18]} [{r['language']}]" for _, r in recs.iterrows()]
        sns.barplot(x=recs["match"], y=labels, color="#2a7f8f", ax=ax[1])
        ax[1].set(xlim=(0, 105), xlabel="Match (percentile in candidate pool)", title="Ranked recommendations")
        savefig("7_live_recommendations.png")


# MAIN

def parse_args():
    ap = argparse.ArgumentParser()
    ap.add_argument("--csv", default="features_30_sec.csv")
    ap.add_argument("--segment", type=float, default=None, help="segment length in seconds (3 for features_3_sec.csv)")
    ap.add_argument("--playlist-dir", default="my_playlist")
    ap.add_argument("--live", action="store_true", help="fetch live recommendations (needs internet + ffmpeg)")
    ap.add_argument("--n-recs", type=int, default=10)
    return ap.parse_known_args()[0]


def load_playlist_dir(folder, segment_sec):
    files = [os.path.join(folder, f) for f in sorted(os.listdir(folder)) if f.lower().endswith((".wav", ".mp3", ".m4a"))]
    if not files:
        return None
    print(f"\n[audio] extracting features from {len(files)} files in '{folder}'")
    return pd.concat([extract_audio_features(f, segment_sec=segment_sec) for f in files], ignore_index=True)


def main():
    args = parse_args()
    raw = load_gtzan(args.csv)
    feature_cols = feature_columns(raw)
    catalog = to_song_level(raw, feature_cols)
    run_eda(catalog)

    ensemble, classes, table, oof_pair = benchmark_genre_models(raw, feature_cols)
    affect = AffectModel().fit(catalog)
    catalog["valence"], catalog["arousal"] = affect.transform(catalog)
    embedder = TasteEmbedder(feature_cols, affect).fit(catalog)
    bundle = Bundle(ensemble, affect, embedder, feature_cols, catalog, classes)
    statistical_audit(catalog, oof_pair)
    plot_validation(catalog, ensemble, feature_cols)
    evaluate_retrieval(catalog, feature_cols, affect)

    playlist_raw = load_playlist_dir(args.playlist_dir, args.segment) if os.path.isdir(args.playlist_dir) else None
    if playlist_raw is not None:
        name = "Your playlist"
    else:
        name = "Demo playlist (GTZAN tracks)"
        ids = pd.concat([catalog[catalog.label == g].sample(n, random_state=SEED)
                         for g, n in zip(["metal", "classical", "pop", "blues"], [6, 5, 5, 4])])["song_id"]
        playlist_raw = raw[raw["song_id"].isin(ids)]
        print("\n[note] demo playlist tracks were seen during training, so their genre probabilities are optimistic.")
    songs = bundle.enrich(playlist_raw)
    profile = build_profile(songs, bundle, name)

    print("\n CATALOG RECOMMENDERS ")
    seed = int((catalog["arousal"] - catalog["valence"]).idxmax())
    print(f"Seed track: {catalog.loc[seed, 'filename']} ({catalog.loc[seed, 'label']})")
    print("\nCosine-similarity neighbours:\n", similar_songs(bundle, seed).to_string(index=False))
    print("\nIso-principle trajectory:\n", mood_trajectory(bundle, seed).to_string(index=False))

    if args.live:
        LiveRecommender(bundle, segment_sec=args.segment).recommend(profile, n_recs=args.n_recs)
    print("\nDone. Figures and feature_table.csv are in ./%s" % OUT_DIR)

import sys
sys.argv = ["x", "--playlist-dir", "/home/abhi/Documents/Projects/DS/my_playlist", "--live", "--n-recs", "15"]

if __name__ == "__main__":
    
    main()


[data] 1000 rows, 1000 songs, 57 base + 38 engineered features

 EDA 
          filename       tempo  rms_mean  spectral_centroid_mean  \
0  blues.00000.wav  123.046875  0.130228             1784.165850   
1  blues.00001.wav   67.999589  0.095948             1530.176679   
2  blues.00002.wav  161.499023  0.175570             1552.811865   
3  blues.00003.wav   63.024009  0.141093             1070.106615   
4  blues.00004.wav  135.999178  0.091529             1835.004266   

   chroma_stft_mean  label  
0          0.350088  blues  
1          0.340914  blues  
2          0.363637  blues  
3          0.404785  blues  
4          0.308526  blues  

Class balance:
 label
blues        100
classical    100
country      100
disco        100
hiphop       100
jazz         100
metal        100
pop          100
reggae       100
rock         100
[saved] outputs/1_eda.png

 GENRE CLASSIFICATION (grouped CV, no song appears in both train and test) 


/home/abhi/Documents/Projects/DS/ds/lib/python3.12/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/abhi/Documents/Projects/DS/ds/lib/python3.12/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/abhi/Documents/Projects/DS/ds/lib/python3.12/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/abhi/Documents/Projects/DS/ds/lib/python3.12/site-packages/sklearn/svm/_base.py:236: FutureWarning: The

                                                                            Model  Accuracy  Macro F1  Kappa    MCC  Fold std
Soft voting (SVM (RBF), MLP (256-128), Logistic Regression, HistGradientBoosting)     0.779    0.7775 0.7544 0.7546    0.0215
                                                                        SVM (RBF)     0.768    0.7683 0.7422 0.7424    0.0172
                                                                    MLP (256-128)     0.749    0.7470 0.7211 0.7213    0.0229
                                                              Logistic Regression     0.732    0.7312 0.7022 0.7023    0.0160
                                                             HistGradientBoosting     0.729    0.7269 0.6989 0.6991    0.0156
                                                                      Extra Trees     0.724    0.7211 0.6933 0.6938    0.0191
                                                                  LDA (shrinkage)     0.723    0.7234 0.6922 0.6925   